In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# STEP 1: LOAD JIGSAW DATA
# ==========================================
print("Loading original Jigsaw dataset...")
df = pd.read_csv('data/train.csv')
df['is_toxic'] = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].max(axis=1)
df = df.dropna(subset=['comment_text'])

# ==========================================
# STEP 2: DATA AUGMENTATION (The Gaming Dictionary)
# ==========================================
print("Injecting custom Gaming Slang dictionary...")

# We create our own mini-dataset of gaming terms. 1 = Toxic, 0 = Friendly
gaming_slang = pd.DataFrame({
    'comment_text': [
        # Toxic Slang
        "gg ez", "tutorial level", "you guys are bots", "sit down", "ff", 
        "free elo", "uninstall", "team diff", "trash team", "smurfs everywhere",
        "ggez", "bot frag", "carried", "iron aim", "diff",
        
        # Friendly Slang (So it doesn't just guess everything is toxic)
        "glhf team!", "nice shot", "ggs", "close one", "nt", 
        "unlucky", "good half", "wp", "nice try", "gr"
    ],
    'is_toxic': [
        # 1s for the toxic words, 0s for the friendly words
        1, 1, 1, 1, 1, 
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 1,
        0, 0, 0, 0, 0, 
        0, 0, 0, 0, 0
    ]
})

# To make sure the model pays attention to our small dictionary, 
# we duplicate (oversample) our slang list a few times before adding it.
gaming_slang_boosted = pd.concat([gaming_slang] * 100, ignore_index=True)

# Merge the boosted gaming slang into the massive Jigsaw dataset
augmented_df = pd.concat([df[['comment_text', 'is_toxic']], gaming_slang_boosted], ignore_index=True)

# ==========================================
# STEP 3: TRAIN ON THE NEW AUGMENTED DATASET
# ==========================================
print("Vectorizing text and training the augmented model...")
X_train, X_test, y_train, y_test = train_test_split(
    augmented_df['comment_text'], augmented_df['is_toxic'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# ==========================================
# STEP 4: THE NEW PRODUCTION FUNCTION
# ==========================================
def calculate_toxicity(chat_text):
    if pd.isna(chat_text) or chat_text == "": return 0.0
    text_features = vectorizer.transform([chat_text])
    return round(model.predict_proba(text_features)[0][1], 4)

print("\n--- NEW LIVE TEST (GAMING SLANG) ---")
print("Text: 'glhf team!' -> Score:", calculate_toxicity("glhf team!"))
print("Text: 'gg ez' -> Score:", calculate_toxicity("gg ez"))
print("Text: 'tutorial level' -> Score:", calculate_toxicity("tutorial level"))
print("Text: 'you guys are bots' -> Score:", calculate_toxicity("you guys are bots"))

Loading original Jigsaw dataset...
Injecting custom Gaming Slang dictionary...
Vectorizing text and training the augmented model...

--- NEW LIVE TEST (GAMING SLANG) ---
Text: 'glhf team!' -> Score: 0.0306
Text: 'gg ez' -> Score: 0.9281
Text: 'tutorial level' -> Score: 0.91
Text: 'you guys are bots' -> Score: 0.9024


In [22]:
import pandas as pd
import random
from tqdm import tqdm
tqdm.pandas()

print("Loading Osama's clean VCT dataset...")
vct_df = pd.read_csv('data/model_ready_win_prediction_dataset.csv')

# ==========================================
# STEP 1: THE PG-13 SYNTHETIC CHAT GENERATOR
# ==========================================
def generate_synthetic_chat(row):
    # Use Pistol Rounds as a safe, leak-free metric for team momentum
    pistol_wins = row['Won_Pistol_Won']

    # Condition 1: Stomping the enemy (Won both pistol rounds)
    if pistol_wins >= 2:
        return random.choice([
            "ez", "sit down", "ff", "you guys are bots", 
            "tutorial level", "gg ez", "free elo", "uninstall"
        ])

    # Condition 2: Getting stomped (Lost both pistol rounds)
    elif pistol_wins == 0:
        return random.choice([
            "you guys are trash", "team diff", "smurfs everywhere", 
            "gg go next", "my team is useless", "report instalock jett", "lag"
        ])

    # Condition 3: Close Game (Split the pistol rounds 1-1)
    else:
        return random.choice([
            "glhf team!", "nice shot", "ggs", "close one", "nt", 
            "unlucky", "good half"
        ])

print("Generating safe-for-school synthetic chat logs...")
vct_df['chat_message'] = vct_df.apply(generate_synthetic_chat, axis=1)

# ==========================================
# STEP 2: GRADE WITH THE AUGMENTED NLP MODEL
# ==========================================
print("Grading the chats with the upgraded NLP brain...")
# This uses the calculate_toxicity function from the previous cell 
vct_df['toxicity_score'] = vct_df['chat_message'].progress_apply(calculate_toxicity)

vct_df['toxicity'] = (vct_df['toxicity_score'] >= 0.5).astype(int)
# ==========================================
# STEP 3: EXPORT FOR XGBOOST
# ==========================================
vct_df.to_csv('data/vct_final_model_ready.csv', index=False)

print("\n🔥 PIPELINE 100% COMPLETE 🔥")
print(f"Final Dataset Shape: {vct_df.shape}")
print(vct_df[['Won_Pistol_Won', 'chat_message', 'toxicity']].head(10))

Loading Osama's clean VCT dataset...
Generating safe-for-school synthetic chat logs...
Grading the chats with the upgraded NLP brain...


100%|██████████| 53914/53914 [00:22<00:00, 2401.18it/s]



🔥 PIPELINE 100% COMPLETE 🔥
Final Dataset Shape: (53914, 37)
   Won_Pistol_Won           chat_message  toxicity
0             2.0      you guys are bots         1
1             1.0                unlucky         0
2             1.0              good half         0
3             2.0         tutorial level         1
4             0.0  report instalock jett         0
5             2.0               free elo         1
6             2.0                  gg ez         1
7             1.0                     nt         0
8             2.0                  gg ez         1
9             0.0             gg go next         1
